<a href="https://colab.research.google.com/github/ealmeida04/logica-programacao/blob/main/preparacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, year, col
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [9]:

# Criando a sessão Spark
spark = SparkSession.builder.appName("YouTubeDataPreparation").getOrCreate()

# 1. Ler o arquivo Parquet

In [12]:
df = spark.read.parquet("/content/videos-comments-tratados.snappy.parquet.txt")
df.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|        0|            8|
|wAZZ-UWGVHI|

2. Adicionar coluna 'Mês' a partir da coluna 'Publicado em'

In [14]:
df = df.withColumn("Mês", month(col("Published At")))
df.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+---+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Mês|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+---+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|  8|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|  8|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|  8|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|        0|    

# 3. Transformar coluna 'keyword' em valores numéricos

In [15]:
indexer = StringIndexer(inputCol="Keyword", outputCol="Keyword_indexed")
df = indexer.fit(df).transform(df)
df.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+---+---------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Mês|Keyword_indexed|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+---+---------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|  8|           17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|  8|           17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|  8|           17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|

# 4. Criar vetor 'Features'

In [19]:
feature_columns = ['Likes', 'Comments', 'Views', 'Interaction', 'Sentiment', 'Likes Comment', 'Mês', 'Keyword_indexed']

# Drop rows with any null values in the feature columns
df = df.na.drop(subset=feature_columns)

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Remove 'features' column if it already exists to avoid IllegalArgumentException on re-run
if "features" in df.columns:
    df = df.drop("features")

df = assembler.transform(df)
df.show(truncate=False)

+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+-------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+---+---------------+--------------------------------------------------------+
|Video ID   |Title                                                                                             |Published At|Keyword|Likes|Comments|Views  |Interaction|Year|Comment                                                                                     

# 5. Normalizar o vetor 'Features'
# Remover valores nulos antes da normalização

In [20]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=False)
scalerModel = scaler.fit(df)
df = scalerModel.transform(df)
df.show(truncate=False)

+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+-------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+---+---------------+--------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Video ID   |Title                                                                                  

# 6. Redução de dimensionalidade com PCA (5 -> 1)

In [21]:
pca = PCA(k=1, inputCol="scaled_features", outputCol="pca_features")
pcaModel = pca.fit(df)
df = pcaModel.transform(df)
df.show(truncate=False)

+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+-------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+---+---------------+--------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+
|Video ID   |Title                                                          

7. Separar em conjuntos de treino (80%) e teste (20%)

In [22]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Tamanho do conjunto de treino: {train_df.count()} linhas")
print(f"Tamanho do conjunto de teste: {test_df.count()} linhas")

Tamanho do conjunto de treino: 12115 linhas
Tamanho do conjunto de teste: 2956 linhas


# 8. Criar modelo de regressão linear para estimar 'Comentários'

In [23]:
lr = LinearRegression(featuresCol='pca_features', labelCol='Comments')
lrModel = lr.fit(train_df)

# Avaliação do modelo

In [24]:
predictions = lrModel.transform(test_df)

evaluator = RegressionEvaluator(
    labelCol="Comments", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE) no conjunto de teste = {rmse}")

evaluator_r2 = RegressionEvaluator(
    labelCol="Comments", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"R-squared (R2) no conjunto de teste = {r2}")

Root Mean Squared Error (RMSE) no conjunto de teste = 14632.129179308427
R-squared (R2) no conjunto de teste = 0.8673005357699304


# 9. Salvar dataframe final em Parquet

In [25]:
df.write.mode("overwrite").parquet("/content/final_dataframe.parquet")
print("DataFrame salvo com sucesso em /content/final_dataframe.parquet")

DataFrame salvo com sucesso em /content/final_dataframe.parquet


Obrigado, espero ter entendido o projeto.